In [1]:
library(H5weaver)
library(Seurat)
library(dplyr)
library(ggplot2)
library(cowplot)
library(stringr)
library(scRepertoire)

Loading required package: data.table

Warning message:
“package ‘data.table’ was built under R version 4.4.3”
Loading required package: Matrix

Loading required package: rhdf5


Attaching package: ‘H5weaver’


The following objects are masked from ‘package:rhdf5’:

    h5dump, h5ls


Loading required package: SeuratObject

Loading required package: sp


Attaching package: ‘SeuratObject’


The following objects are masked from ‘package:base’:

    intersect, t


Warning message:
“package ‘dplyr’ was built under R version 4.4.3”

Attaching package: ‘dplyr’


The following objects are masked from ‘package:data.table’:

    between, first, last


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union


Warning message:
“package ‘ggplot2’ was built under R version 4.4.3”
Warning message:
“package ‘stringr’ was built under R version 4.4.3”


In [2]:
so <- readRDS('vst3_t_cells_clustered.rds')
so

An object of class Seurat 
18129 features across 856496 samples within 1 assay 
Active assay: RNA (18129 features, 2000 variable features)
 89 layers present: counts.1.1.1.1.1.1.1.1.1.1.1.1.1.1.1.1.1.1.1.1.1.1.1.1.1.1.1.1.1.1.1.1.1.1.1.1.1.1.1.1.1.1.1, counts.2, counts.2.1, counts.2.1.1, counts.2.1.1.1, counts.2.1.1.1.1, counts.2.1.1.1.1.1, counts.2.1.1.1.1.1.1, counts.2.1.1.1.1.1.1.1, counts.2.1.1.1.1.1.1.1.1, counts.2.1.1.1.1.1.1.1.1.1, counts.2.1.1.1.1.1.1.1.1.1.1, counts.2.1.1.1.1.1.1.1.1.1.1.1, counts.2.1.1.1.1.1.1.1.1.1.1.1.1, counts.2.1.1.1.1.1.1.1.1.1.1.1.1.1, counts.2.1.1.1.1.1.1.1.1.1.1.1.1.1.1, counts.2.1.1.1.1.1.1.1.1.1.1.1.1.1.1.1, counts.2.1.1.1.1.1.1.1.1.1.1.1.1.1.1.1.1, counts.2.1.1.1.1.1.1.1.1.1.1.1.1.1.1.1.1.1, counts.2.1.1.1.1.1.1.1.1.1.1.1.1.1.1.1.1.1.1, counts.2.1.1.1.1.1.1.1.1.1.1.1.1.1.1.1.1.1.1.1, counts.2.1.1.1.1.1.1.1.1.1.1.1.1.1.1.1.1.1.1.1.1, counts.2.1.1.1.1.1.1.1.1.1.1.1.1.1.1.1.1.1.1.1.1.1, counts.2.1.1.1.1.1.1.1.1.1.1.1.1.1.1.1.1.1.1.1.1.1.1, counts.2.1.

In [ ]:
# so <- JoinLayers(so)

In [3]:
table(so$AIFI_PBMC.Flex_L1)


T cell 
856496 

In [4]:
table(so$pool_id)


EXP-01618-PA EXP-01618-PB EXP-01618-PC EXP-01618-PD 
      186145       229945       215152       225254 

In [5]:
table(so$well_id)


EXP-01618-PAC1W1 EXP-01618-PAC1W2 EXP-01618-PBC1W3 EXP-01618-PBC1W4 
           84066           102079           113175           116770 
EXP-01618-PCC2W1 EXP-01618-PCC2W2 EXP-01618-PDC2W3 EXP-01618-PDC2W4 
          109487           105665           113325           111929 

In [6]:
table(so$sample_id)


FIX10434-016 FIX10434-017 FIX10434-018 FIX10434-019 FIX10434-020 FIX10434-021 
       22343        27155        20657        22893        27169        28999 
FIX10434-022 FIX10434-023 FIX10434-024 FIX10434-025 FIX10434-026 FIX10434-027 
        7073        10569         7192         6971        13721        14951 
FIX10434-028 FIX10434-029 FIX10434-030 FIX10434-031 FIX10434-032 FIX10434-033 
        5573        12412        27956        13782        18780        22612 
FIX10434-035 FIX10434-036 FIX10434-037 FIX10434-038 FIX10435-016 FIX10435-017 
       13180        31293        20281        36228        31223        32467 
FIX10435-018 FIX10435-019 FIX10435-020 FIX10435-021 FIX10435-022 FIX10435-023 
       34320        29592        28972        33528        15567        15679 
FIX10435-024 FIX10435-025 FIX10435-026 FIX10435-027 FIX10435-028 FIX10435-029 
       14684        13109        14208        14764        18340        21272 
FIX10435-030 FIX10435-031 FIX10435-032 FIX10435-033

## Add TCR

In [7]:
add_tcr_func_sample <- function(seurat_object, pool){
    # build barcode dataframe
    barcodes_df <- select(seurat_object@meta.data, classic_barcodes, barcodes)

    # extract sample ID
    sample_id <- unique(seurat_object$sample_id)
    print(sample_id)
    flush.console()
    
    # load tcr
    tcr_file_path <- list.files(path = paste0('tcr/',pool), pattern = paste0(sample_id,'.clones.tsv'), full.names = T)
    # print(tcr_file_path)
    tcr <- read.csv(tcr_file_path, sep='\t')

    # replace barcodes with barcodes from pipeline
    tcr_updated <- tcr %>%
        left_join(barcodes_df, by = c("tagValueCELL" = "classic_barcodes")) %>%
        mutate(tagValueCELL = ifelse(!is.na(barcodes), barcodes, tagValueCELL)) %>%
        select(-barcodes)
    # trim barcodes for any non-TCR clones
    tcr_updated <- tcr_updated[!(tcr_updated$topChains %in% c("IGH", "IGK", "IGL")), ]
    # load via scRep
    tcr_contigs <- loadContigs(tcr_updated, format = 'MiXCR')
    combined.TCR <- combineTCR(tcr_contigs,
                               removeNA = FALSE, 
                               removeMulti = FALSE, 
                               filterMulti = TRUE)
    seurat_object <- combineExpression(combined.TCR, 
                                 seurat_object, 
                                 cloneCall="gene",
                                 proportion = TRUE)

    return(seurat_object)
}

In [11]:
add_tcr_func_batch <- function(seurat_object,pool){
    # split object by sample
    print("Splitting Object")
    flush.console()
    so_split <- SplitObject(seurat_object, split.by = 'sample_id')
    # loop through and add tcr info
    print("Adding TCR")
    flush.console()
    meta_split <- lapply(so_split, function(x){
        x <- add_tcr_func_sample(x, pool)
        return(x@meta.data)
    })
    print("Merging Metadata")
    flush.console()
    meta_bind <- bind_rows(meta_split)    
    return(meta_bind)
}

In [9]:
so$classic_barcodes <- substr(so$original_barcodes, 1, nchar(so$original_barcodes)-8)

In [10]:
well_split <- SplitObject(so, split.by = 'well_id')

In [12]:
well_split$`EXP-01618-PAC1W1` <- add_tcr_func_batch(well_split$`EXP-01618-PAC1W1`, 'P1A_tcr')

[1] "Splitting Object"
[1] "Adding TCR"
[1] "FIX10434-016"
[1] "FIX10434-026"
[1] "FIX10434-027"
[1] "FIX10434-023"
[1] "FIX10434-021"
[1] "FIX10434-020"
[1] "FIX10434-024"
[1] "FIX10434-038"
[1] "FIX10434-018"
[1] "FIX10435-030"
[1] "FIX10434-036"
[1] "FIX10434-035"
[1] "FIX10434-025"
[1] "FIX10434-022"
[1] "FIX10434-017"
[1] "FIX10434-019"
[1] "Merging Metadata"


In [14]:
well_split$`EXP-01618-PAC1W2` <- add_tcr_func_batch(well_split$`EXP-01618-PAC1W2`, 'P1B_tcr')

[1] "Splitting Object"
[1] "Adding TCR"
[1] "FIX10434-016"
[1] "FIX10434-026"
[1] "FIX10434-027"
[1] "FIX10434-023"
[1] "FIX10434-021"
[1] "FIX10434-020"
[1] "FIX10434-024"
[1] "FIX10434-038"
[1] "FIX10434-018"
[1] "FIX10435-030"
[1] "FIX10434-036"
[1] "FIX10434-035"
[1] "FIX10434-025"
[1] "FIX10434-022"
[1] "FIX10434-017"
[1] "FIX10434-019"
[1] "Merging Metadata"


In [15]:
well_split$`EXP-01618-PBC1W3` <- add_tcr_func_batch(well_split$`EXP-01618-PBC1W3`, 'P2A_tcr')
well_split$`EXP-01618-PBC1W4` <- add_tcr_func_batch(well_split$`EXP-01618-PBC1W4`, 'P2B_tcr')

[1] "Splitting Object"
[1] "Adding TCR"
[1] "FIX10434-029"
[1] "FIX10434-016"
[1] "FIX10434-021"
[1] "FIX10434-020"
[1] "FIX10434-038"
[1] "FIX10434-032"
[1] "FIX10434-018"
[1] "FIX10435-030"
[1] "FIX10434-031"
[1] "FIX10434-030"
[1] "FIX10434-037"
[1] "FIX10434-036"
[1] "FIX10434-033"
[1] "FIX10434-017"
[1] "FIX10434-019"
[1] "FIX10434-028"
[1] "Merging Metadata"
[1] "Splitting Object"
[1] "Adding TCR"
[1] "FIX10434-029"
[1] "FIX10434-016"
[1] "FIX10434-021"
[1] "FIX10434-020"
[1] "FIX10434-038"
[1] "FIX10434-032"
[1] "FIX10434-018"
[1] "FIX10435-030"
[1] "FIX10434-031"
[1] "FIX10434-030"
[1] "FIX10434-037"
[1] "FIX10434-036"
[1] "FIX10434-033"
[1] "FIX10434-017"
[1] "FIX10434-019"
[1] "FIX10434-028"
[1] "Merging Metadata"


In [16]:
well_split$`EXP-01618-PCC2W1` <- add_tcr_func_batch(well_split$`EXP-01618-PCC2W1`, 'P3A_tcr')
well_split$`EXP-01618-PCC2W2` <- add_tcr_func_batch(well_split$`EXP-01618-PCC2W2`, 'P3B_tcr')

[1] "Splitting Object"
[1] "Adding TCR"
[1] "FIX10435-028"
[1] "FIX10435-020"
[1] "FIX10435-017"
[1] "FIX10435-021"
[1] "FIX10435-023"
[1] "FIX10435-031"
[1] "FIX10435-024"
[1] "FIX10435-026"
[1] "FIX10435-019"
[1] "FIX10435-018"
[1] "FIX10435-016"
[1] "FIX10435-025"
[1] "FIX10435-027"
[1] "FIX10434-038"
[1] "FIX10435-022"
[1] "FIX10435-029"
[1] "Merging Metadata"
[1] "Splitting Object"
[1] "Adding TCR"
[1] "FIX10435-028"
[1] "FIX10435-020"
[1] "FIX10435-017"
[1] "FIX10435-021"
[1] "FIX10435-023"
[1] "FIX10435-031"
[1] "FIX10435-024"
[1] "FIX10435-026"
[1] "FIX10435-019"
[1] "FIX10435-018"
[1] "FIX10435-016"
[1] "FIX10435-025"
[1] "FIX10435-027"
[1] "FIX10434-038"
[1] "FIX10435-022"
[1] "FIX10435-029"
[1] "Merging Metadata"


In [19]:
well_split$`EXP-01618-PDC2W4` <- add_tcr_func_batch(well_split$`EXP-01618-PDC2W4`, 'P4B_tcr')

[1] "Splitting Object"
[1] "Adding TCR"
[1] "FIX10435-028"
[1] "FIX10435-020"
[1] "FIX10435-017"
[1] "FIX10435-034"
[1] "FIX10435-021"
[1] "FIX10435-032"
[1] "FIX10435-031"
[1] "FIX10435-033"
[1] "FIX10435-019"
[1] "FIX10435-018"
[1] "FIX10435-016"
[1] "FIX10434-038"
[1] "FIX10435-037"
[1] "FIX10435-035"
[1] "FIX10435-029"
[1] "FIX10435-036"
[1] "Merging Metadata"


In [18]:
well_split$`EXP-01618-PDC2W3` <- add_tcr_func_batch(well_split$`EXP-01618-PDC2W3`, 'P4A_tcr')

[1] "Splitting Object"
[1] "Adding TCR"
[1] "FIX10435-028"
[1] "FIX10435-020"
[1] "FIX10435-017"
[1] "FIX10435-034"
[1] "FIX10435-021"
[1] "FIX10435-032"
[1] "FIX10435-031"
[1] "FIX10435-033"
[1] "FIX10435-019"
[1] "FIX10435-018"
[1] "FIX10435-016"
[1] "FIX10434-038"
[1] "FIX10435-037"
[1] "FIX10435-035"
[1] "FIX10435-029"
[1] "FIX10435-036"
[1] "Merging Metadata"


In [20]:
well_meta_merge <- bind_rows(well_split)
nrow(well_meta_merge)

[1] 856496

In [21]:
ncol(so[['RNA']])

[1] 856496

In [26]:
colnames(well_meta_merge)

[1] "orig.ident"        "nCount_RNA"        "nFeature_RNA"     
 [4] "barcodes"          "AIFI_PBMC.Flex_L1" "AIFI_PBMC.Flex_L2"
 [7] "AIFI_PBMC.Flex_L3" "batch_id"          "cell_name"        
[10] "cell_uuid"         "chip_id"           "n_genes"          
[13] "n_reads"           "n_umis"            "original_barcodes"
[16] "pool_id"           "probe_barcode"     "sample_id"        
[19] "scrublet_call"     "scrublet_score"    "well_id"          
[22] "Donor"             "Day"               "Culture"          
[25] "Peptide"           "Stemness"          "Concentration"    
[28] "RNA_snn_res.1"     "seurat_clusters"   "classic_barcodes" 
[31] "CTgene"            "CTnt"              "CTaa"             
[34] "CTstrict"          "clonalProportion"  "clonalFrequency"  
[37] "cloneSize"

In [28]:
well_meta_trim <- select(well_meta_merge, barcodes, CTgene, CTnt, CTaa, CTstrict, clonalProportion, clonalFrequency, cloneSize)

# Add Metadata

In [29]:
new_meta <- so@meta.data %>%
    left_join(well_meta_trim, by = 'barcodes')

In [30]:
table(so$barcodes == new_meta$barcodes)


  TRUE 
856496 

In [31]:
head(new_meta)
colnames(new_meta)

,orig.ident,nCount_RNA,nFeature_RNA,barcodes,AIFI_PBMC.Flex_L1,AIFI_PBMC.Flex_L2,AIFI_PBMC.Flex_L3,batch_id,cell_name,cell_uuid,⋯,RNA_snn_res.1,seurat_clusters,classic_barcodes,CTgene,CTnt,CTaa,CTstrict,clonalProportion,clonalFrequency,cloneSize
,<chr>,<dbl>,<int>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,⋯,<fct>,<fct>,<chr>,<chr>,<chr>,<chr>,<chr>,<dbl>,<int>,<fct>
1,SeuratProject,1857,1393,af33653c7ebf11f0b8e5ee35decf83ac,T cell,Memory CD4 T cell,GZMB- CD27+ EM CD4 T cell,EXP-01618-PC,associated_archaic_koodoo,af33653c7ebf11f0b8e5ee35decf83ac,⋯,2,2,AAACAAGCAAACTTGG,TRAV13-1*00(1277.8).TRAJ33*00(544.7).TRAC*00(65)_TRBV11-2*00(850.3).TRBD2*00(35).TRBJ2-3*00(424.2).TRBC2*00(296.4),TGTGCAGCGGATAGCAACTATCAGTTAATCTGG_TGTGCCAGCAGGAGGGCCGGTGATACGCAGTATTTT,CAADSNYQLIW_CASRRAGDTQYF,TRAV13-1*00(1277.8).TRAJ33*00(544.7).TRAC*00(65);TGTGCAGCGGATAGCAACTATCAGTTAATCTGG_TRBV11-2*00(850.3).TRBD2*00(35).TRBJ2-3*00(424.2).TRBC2*00(296.4);TGTGCCAGCAGGAGGGCCGGTGATACGCAGTATTTT,0.0002643405,1,Small (1e-04 < X <= 0.001)
2,SeuratProject,1221,1018,af3367b27ebf11f0b8e5ee35decf83ac,T cell,Naive CD4 T cell,Core naive CD4 T cell,EXP-01618-PC,antiviral_earth_whapuku,af3367b27ebf11f0b8e5ee35decf83ac,⋯,1,1,AAACAAGCACCTTCTC,"NA_TRBV12-3*00(788.6),TRBV12-4*00(788.6).TRBD1*00(25).TRBJ1-2*00(438).TRBC1*00(335.9)",NA_TGTGCCAGCAGTTTAGTAGGGGTCTATGGCTACACCTTC,NA_CASSLVGVYGYTF,"NA;NA_TRBV12-3*00(788.6),TRBV12-4*00(788.6).TRBD1*00(25).TRBJ1-2*00(438).TRBC1*00(335.9);TGTGCCAGCAGTTTAGTAGGGGTCTATGGCTACACCTTC",0.0002643405,1,Small (1e-04 < X <= 0.001)
3,SeuratProject,6182,3224,af33691a7ebf11f0b8e5ee35decf83ac,T cell,DN T cell,DN T cell,EXP-01618-PC,unnoted_ageold_bactrian,af33691a7ebf11f0b8e5ee35decf83ac,⋯,14,14,AAACCAATCACAGGAT,TRAV16*00(882.2).TRAJ61*00(580.8).TRAC*00(221)_TRBV28*00(953.1).TRBD2*00(45).TRBJ2-5*00(453.8).TRBC2*00(198),TGTGCTCTAAGTCTGTACCGGGTTAATAGGAAACTGACATTT_TGTGCCAGCAGCACTAGCGGGGGCCATCAAGAGACCCAGTACTTC,CALSLYRVNRKLTF_CASSTSGGHQETQYF,TRAV16*00(882.2).TRAJ61*00(580.8).TRAC*00(221);TGTGCTCTAAGTCTGTACCGGGTTAATAGGAAACTGACATTT_TRBV28*00(953.1).TRBD2*00(45).TRBJ2-5*00(453.8).TRBC2*00(198);TGTGCCAGCAGCACTAGCGGGGGCCATCAAGAGACCCAGTACTTC,0.0002643405,1,Small (1e-04 < X <= 0.001)
4,SeuratProject,6421,3367,af336a467ebf11f0b8e5ee35decf83ac,T cell,Memory CD8 T cell,Proliferating T cell,EXP-01618-PC,authorial_crazed_bunny,af336a467ebf11f0b8e5ee35decf83ac,⋯,15,15,AAACCAATCACGGACA,TRAV13-1*00(1305).TRAJ45*00(512.8)._NA,TGTGCAGCATTGGGGCTCGTTTGGGGAGGTGCTGACGGACTCACCTTT_NA,CAALGLVWGGADGLTF_NA,TRAV13-1*00(1305).TRAJ45*00(512.8).;TGTGCAGCATTGGGGCTCGTTTGGGGAGGTGCTGACGGACTCACCTTT_NA;NA,0.0002643405,1,Small (1e-04 < X <= 0.001)
5,SeuratProject,3403,2179,af336b4a7ebf11f0b8e5ee35decf83ac,T cell,Naive CD8 T cell,ISG+ naive CD8 T cell,EXP-01618-PC,crimpy_cibophobic_gnat,af336b4a7ebf11f0b8e5ee35decf83ac,⋯,0,0,AAACCAATCATGAATG,NA,NA,NA,NA,NA,NA,NA
6,SeuratProject,4009,2366,af336c587ebf11f0b8e5ee35decf83ac,T cell,Proliferating T cell,Proliferating T cell,EXP-01618-PC,only_lacquer_antelope,af336c587ebf11f0b8e5ee35decf83ac,⋯,5,5,AAACCAATCCAAAGGT,"TRAV8-3*00(1014.7).TRAJ7*00(550.3).TRAC*00(169.3)_TRBV5-8*00(633.6).TRBD1*00(30),TRBD2*00(30).TRBJ2-3*00(407.8).TRBC2*00(336.3)",TGTGCTGTGGCCGCCTATGGGAACAACAGACTCGCTTTT_TGTGCCAGCAGCCCCGGGGGAAGCCTATATTCACAGAGTACGCAGTATTTT,CAVAAYGNNRLAF_CASSPGGSLYSQSTQYF,"TRAV8-3*00(1014.7).TRAJ7*00(550.3).TRAC*00(169.3);TGTGCTGTGGCCGCCTATGGGAACAACAGACTCGCTTTT_TRBV5-8*00(633.6).TRBD1*00(30),TRBD2*00(30).TRBJ2-3*00(407.8).TRBC2*00(336.3);TGTGCCAGCAGCCCCGGGGGAAGCCTATATTCACAGAGTACGCAGTATTTT",0.0095162569,36,Medium (0.001 < X <= 0.01)


[1] "orig.ident"        "nCount_RNA"        "nFeature_RNA"     
 [4] "barcodes"          "AIFI_PBMC.Flex_L1" "AIFI_PBMC.Flex_L2"
 [7] "AIFI_PBMC.Flex_L3" "batch_id"          "cell_name"        
[10] "cell_uuid"         "chip_id"           "n_genes"          
[13] "n_reads"           "n_umis"            "original_barcodes"
[16] "pool_id"           "probe_barcode"     "sample_id"        
[19] "scrublet_call"     "scrublet_score"    "well_id"          
[22] "Donor"             "Day"               "Culture"          
[25] "Peptide"           "Stemness"          "Concentration"    
[28] "RNA_snn_res.1"     "seurat_clusters"   "classic_barcodes" 
[31] "CTgene"            "CTnt"              "CTaa"             
[34] "CTstrict"          "clonalProportion"  "clonalFrequency"  
[37] "cloneSize"

In [32]:
so$CTgene <- new_meta$CTgene
so$CTnt <- new_meta$CTnt
so$CTaa <- new_meta$CTaa
so$CTstrict <- new_meta$CTstrict
so$clonalProportion <- new_meta$clonalProportion
so$clonalFrequency <- new_meta$clonalFrequency
so$cloneSize <- new_meta$cloneSize

In [35]:
so$pep_enriched <- 'Non-Activated'
so$pep_enriched[so$`RNA_snn_res.1` %in% c(15,22,29,5,8)] <- 'Pep-Enriched'

In [48]:
saveRDS(so, 'vst3_with_tcr.rds')